In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
STEP1_figures_tables.py
-----------------------
Create main-text and appendix figures/tables for STEP1 (Distance & MDS)
from precomputed R outputs (.rds, .csv).

Usage example:
    python STEP1_figures_tables.py \
        --root ~/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127
"""

import os
import argparse
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadr


# =============
# 0. Matplotlib style
# =============

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "figure.figsize": (7.5, 5.0),
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})


# =============
# 1. Helper functions
# =============

def find_single_file(directory: str, pattern: str) -> str:
    """
    Find a single file whose name matches the given pattern (glob).
    If multiple files are found, the first one is returned.
    Pattern例: 'distance_raw_*.rds'
    """
    import glob
    search_pattern = os.path.join(directory, pattern)
    files = sorted(glob.glob(search_pattern))
    if len(files) == 0:
        raise FileNotFoundError(f"No file found in {directory} with pattern: {pattern}")
    if len(files) > 1:
        print(f"[WARN] Multiple files found in {directory} with pattern: {pattern}")
        print(f"       Using the first one: {os.path.basename(files[0])}")
    return files[0]


def read_rds_single(path: str):
    """
    Read a .rds file using pyreadr and return the single object.
    Many .rds files in this project are a single object (matrix, dist, data.frame, etc.).
    """
    result = pyreadr.read_r(path)
    if len(result.keys()) != 1:
        # If multiple objects exist, just take the first (can be customized if needed)
        print(f"[WARN] Multiple objects in {path}, using the first one.")
    key = list(result.keys())[0]
    return result[key]


def distance_to_vector(obj) -> np.ndarray:
    """
    Convert an R distance object (read via pyreadr) to a 1D numpy array of distances.
    - If obj is a pandas Series or 1D array: return as 1D.
    - If obj is a DataFrame or 2D array: use the upper triangle (excluding diagonal).
    """
    # pandas DataFrame
    if isinstance(obj, pd.DataFrame):
        mat = obj.values
    # pandas Series
    elif isinstance(obj, pd.Series):
        return obj.to_numpy().astype(float)
    # numpy array
    elif isinstance(obj, np.ndarray):
        if obj.ndim == 1:
            return obj.astype(float)
        elif obj.ndim == 2:
            mat = obj
        else:
            raise ValueError("Unsupported array dimension for distance object.")
    else:
        # Other types (e.g., scalar) -> try to convert directly
        arr = np.array(obj)
        if arr.ndim == 1:
            return arr.astype(float)
        elif arr.ndim == 2:
            mat = arr
        else:
            raise ValueError("Unknown distance object type.")

    # Square matrix -> take upper triangle without diagonal
    if mat.shape[0] == mat.shape[1]:
        iu = np.triu_indices_from(mat, k=1)
        v = mat[iu]
        return v.astype(float)
    else:
        # Non-square -> flatten all
        return mat.astype(float).ravel()


def read_distance_pair(step1_dir: str, dataset_folder: str) -> dict:
    """
    Read raw and corrected distance matrices for one dataset.
    Returns dict: {'raw': obj_raw, 'corr': obj_corr}
    """
    ddir = os.path.join(step1_dir, dataset_folder)
    raw_file = find_single_file(ddir, "distance_raw_*.rds")
    corr_file = find_single_file(ddir, "distance_corr_*.rds")
    print(f"[INFO] Reading raw distance from: {raw_file}")
    print(f"[INFO] Reading corrected distance from: {corr_file}")
    raw = read_rds_single(raw_file)
    corr = read_rds_single(corr_file)
    return {"raw": raw, "corr": corr}


def read_cmdscale_mds(step1_dir: str, dataset_label: str, dataset_folder: str) -> pd.DataFrame:
    """
    Read cmdscale MDS coordinates from .rds.
    Assumes the .rds contains a matrix or data frame with columns = MDS dimensions.
    """
    ddir = os.path.join(step1_dir, dataset_folder)
    pattern = f"mds_cmdscale_{dataset_label}_*.rds"
    rds_file = find_single_file(ddir, pattern)
    print(f"[INFO] Reading cmdscale MDS for {dataset_label} from: {rds_file}")
    obj = read_rds_single(rds_file)

    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    else:
        arr = np.array(obj)
        df = pd.DataFrame(arr)

    if df.shape[1] < 2:
        raise ValueError(f"cmdscale MDS object for {dataset_label} has < 2 columns.")

    df = df.iloc[:, :2].copy()
    df.columns = ["MDS1", "MDS2"]
    return df


def read_isomds_mds(step1_dir: str, dataset_label: str, dataset_folder: str) -> pd.DataFrame:
    """
    Read isoMDS coordinates from .rds.
    Here we assume that the .rds contains either:
      - a matrix / data.frame with coordinates, or
      - a list-like object with a 'points' component already saved separately.
    If your isoMDS result is a list with $points, please export the coordinates
    as a matrix/data.frame in R when saving the .rds.
    """
    ddir = os.path.join(step1_dir, dataset_folder)
    pattern = f"mds_isoMDS_{dataset_label}_*.rds"
    rds_file = find_single_file(ddir, pattern)
    print(f"[INFO] Reading isoMDS for {dataset_label} from: {rds_file}")
    obj = read_rds_single(rds_file)

    # If already DataFrame/matrix-like
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    else:
        arr = np.array(obj)
        if arr.ndim == 2:
            df = pd.DataFrame(arr)
        else:
            # As a fallback, treat as one big vector (not expected)
            df = pd.DataFrame(arr.reshape(-1, 2))

    if df.shape[1] < 2:
        raise ValueError(f"isoMDS object for {dataset_label} has < 2 columns.")

    df = df.iloc[:, :2].copy()
    df.columns = ["MDS1", "MDS2"]
    return df


def read_cmdscale_eigvals(step1_dir: str, dataset_label: str, dataset_folder: str) -> pd.DataFrame:
    """
    Read cmdscale eigenvalues from CSV.
    We assume the CSV has either:
      - columns 'dim' and 'eigenvalue', or
      - some numeric column for eigenvalue, and 'dim' is row number.
    """
    ddir = os.path.join(step1_dir, dataset_folder)
    pattern = f"eigvals_cmdscale_{dataset_label}_*.csv"
    csv_file = find_single_file(ddir, pattern)
    print(f"[INFO] Reading eigenvalues for {dataset_label} from: {csv_file}")
    df = pd.read_csv(csv_file)

    # Try to find eigenvalue column
    if "eigenvalue" not in df.columns:
        num_cols = df.select_dtypes(include=[np.number])
        if num_cols.shape[1] == 0:
            raise ValueError(f"No numeric column found in eigenvalue CSV for {dataset_label}")
        ev_col = num_cols.columns[0]
        df["eigenvalue"] = num_cols[ev_col]

    if "dim" not in df.columns:
        df["dim"] = np.arange(1, len(df) + 1)

    df = df[["dim", "eigenvalue"]].copy()
    df = df.sort_values("dim")
    return df


def read_isomds_stress(step1_dir: str, dataset_label: str, dataset_folder: str) -> float:
    """
    Read isoMDS stress.
    We first look for 'stress_isoMDS_<dataset>_*.rds',
    then for 'stress_isoMDS_<dataset>_*.csv'.
    """
    ddir = os.path.join(step1_dir, dataset_folder)

    # .rds
    try:
        rds_file = find_single_file(ddir, f"stress_isoMDS_{dataset_label}_*.rds")
        print(f"[INFO] Reading isoMDS stress for {dataset_label} from: {rds_file}")
        obj = read_rds_single(rds_file)
        arr = np.array(obj).astype(float).ravel()
        return float(arr[0])
    except FileNotFoundError:
        pass

    # .csv
    try:
        csv_file = find_single_file(ddir, f"stress_isoMDS_{dataset_label}_*.csv")
        print(f"[INFO] Reading isoMDS stress for {dataset_label} from: {csv_file}")
        df = pd.read_csv(csv_file)
        num_cols = df.select_dtypes(include=[np.number])
        if num_cols.shape[1] == 0:
            raise ValueError(f"No numeric column found in stress CSV for {dataset_label}")
        # Use the first numeric column's mean as stress
        return float(num_cols.iloc[:, 0].mean())
    except FileNotFoundError:
        print(f"[WARN] No stress_isoMDS file found for dataset {dataset_label}, returning NaN.")
        return float("nan")


# =============
# 2. Plot functions
# =============

def plot_distance_distribution_AB(dist_A: dict, dist_B: dict,
                                  outfile_pdf: str, outfile_png: str = None):
    """
    Fig. 4.2A - Signless correlation distance distribution (raw vs corrected) for A/B.
    """
    vA_raw = distance_to_vector(dist_A["raw"])
    vA_corr = distance_to_vector(dist_A["corr"])
    vB_raw = distance_to_vector(dist_B["raw"])
    vB_corr = distance_to_vector(dist_B["corr"])

    df_A = pd.DataFrame({
        "distance": np.concatenate([vA_raw, vA_corr]),
        "type": (["raw"] * len(vA_raw)) + (["corrected"] * len(vA_corr)),
        "dataset": "A (OH + others)"
    })
    df_B = pd.DataFrame({
        "distance": np.concatenate([vB_raw, vB_corr]),
        "type": (["raw"] * len(vB_raw)) + (["corrected"] * len(vB_corr)),
        "dataset": "B (FP + others)"
    })
    df_all = pd.concat([df_A, df_B], ignore_index=True)

    # Histogram with facets by dataset
    g = sns.displot(
        data=df_all,
        x="distance",
        hue="type",
        col="dataset",
        kind="hist",
        bins=60,
        element="step",
        stat="count",
        common_norm=False,
        facet_kws=dict(sharex=True, sharey=False),
        height=4.0,
        aspect=1.1,
        alpha=0.5
    )
    g.set(xlim=(0, 2))
    g.set_axis_labels("Signless correlation distance", "Frequency")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle("Signless correlation distance distribution (raw vs corrected)")

    g.savefig(outfile_pdf)
    print(f"[SAVED] Fig. 4.2A (PDF): {outfile_pdf}")
    if outfile_png is not None:
        g.savefig(outfile_png)
        print(f"[SAVED] Fig. 4.2A (PNG): {outfile_png}")
    plt.close(g.fig)


def plot_cmdscale_MDS_AB(mds_A: pd.DataFrame, mds_B: pd.DataFrame, outfile_pdf: str):
    """
    Fig. 4.2B - Classical MDS (cmdscale) 2D map for A/B.
    """
    df_A = mds_A.copy()
    df_A["dataset"] = "A (OH + others)"
    df_B = mds_B.copy()
    df_B["dataset"] = "B (FP + others)"
    df_all = pd.concat([df_A, df_B], ignore_index=True)

    g = sns.relplot(
        data=df_all,
        x="MDS1", y="MDS2",
        col="dataset",
        kind="scatter",
        alpha=0.6,
        s=6,
        facet_kws=dict(sharex=False, sharey=False),
        height=4.0,
        aspect=1.1
    )
    g.set_axis_labels("MDS1", "MDS2")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle("Classical MDS (cmdscale) maps based on corrected distances")

    g.savefig(outfile_pdf)
    print(f"[SAVED] Fig. 4.2B: {outfile_pdf}")
    plt.close(g.fig)


def plot_isoMDS_AB(mds_A: pd.DataFrame, mds_B: pd.DataFrame, outfile_pdf: str):
    """
    Fig. 4.2C - Non-metric MDS (isoMDS) 2D map for A/B.
    """
    df_A = mds_A.copy()
    df_A["dataset"] = "A (OH + others)"
    df_B = mds_B.copy()
    df_B["dataset"] = "B (FP + others)"
    df_all = pd.concat([df_A, df_B], ignore_index=True)

    g = sns.relplot(
        data=df_all,
        x="MDS1", y="MDS2",
        col="dataset",
        kind="scatter",
        alpha=0.6,
        s=6,
        facet_kws=dict(sharex=False, sharey=False),
        height=4.0,
        aspect=1.1
    )
    g.set_axis_labels("MDS1", "MDS2")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle("Non-metric MDS (isoMDS) maps based on corrected distances")

    g.savefig(outfile_pdf)
    print(f"[SAVED] Fig. 4.2C: {outfile_pdf}")
    plt.close(g.fig)


def plot_scree_AB(eig_A: pd.DataFrame, eig_B: pd.DataFrame, outfile_pdf: str):
    """
    Fig. 4.2D - Scree plot of cmdscale eigenvalues (A/B).
    """
    df_A = eig_A.copy()
    df_A["dataset"] = "A (OH + others)"
    df_B = eig_B.copy()
    df_B["dataset"] = "B (FP + others)"
    df_all = pd.concat([df_A, df_B], ignore_index=True)

    g = sns.relplot(
        data=df_all,
        x="dim", y="eigenvalue",
        col="dataset",
        kind="line",
        marker="o",
        facet_kws=dict(sharex=True, sharey=False),
        height=4.0,
        aspect=1.1
    )
    g.set_axis_labels("Dimension", "Eigenvalue")
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle("Scree plots of cmdscale eigenvalues")

    g.savefig(outfile_pdf)
    print(f"[SAVED] Fig. 4.2D: {outfile_pdf}")
    plt.close(g.fig)


def plot_scree_single(eig_df: pd.DataFrame, dataset_title: str, outfile_pdf: str):
    """
    Scree plot for a single dataset (appendix).
    """
    fig, ax = plt.subplots()
    ax.plot(eig_df["dim"], eig_df["eigenvalue"], marker="o", linewidth=1.5)
    ax.set_xlabel("Dimension")
    ax.set_ylabel("Eigenvalue")
    ax.set_title(f"Scree plot ({dataset_title})")
    fig.tight_layout()
    fig.savefig(outfile_pdf)
    print(f"[SAVED] Scree plot (appendix): {outfile_pdf}")
    plt.close(fig)


def plot_cmdscale_single(mds_df: pd.DataFrame, dataset_title: str, outfile_pdf: str):
    """
    2D MDS map for a single dataset (appendix).
    """
    fig, ax = plt.subplots()
    ax.scatter(mds_df["MDS1"], mds_df["MDS2"], alpha=0.6, s=6)
    ax.set_xlabel("MDS1")
    ax.set_ylabel("MDS2")
    ax.set_title(f"Classical MDS map ({dataset_title})")
    fig.tight_layout()
    fig.savefig(outfile_pdf)
    print(f"[SAVED] MDS map (appendix): {outfile_pdf}")
    plt.close(fig)


# =============
# 3. Table creation
# =============

def make_table_distance_stats(dist_by_ds: dict) -> pd.DataFrame:
    """
    Table S4.2.1 - raw vs corrected distance statistics (A/B/C/D).
    dist_by_ds: dict like {"A": {"raw": obj, "corr": obj}, ...}
    """
    rows = []
    for ds_label, dl in dist_by_ds.items():
        for tp in ["raw", "corrected"]:
            v = distance_to_vector(dl["raw"] if tp == "raw" else dl["corr"])
            v = v[~np.isnan(v)]
            q1, median, q3 = np.quantile(v, [0.25, 0.5, 0.75])
            row = {
                "dataset": ds_label,
                "type": tp,
                "mean": float(np.mean(v)),
                "median": float(median),
                "sd": float(np.std(v, ddof=1)),
                "min": float(np.min(v)),
                "q1": float(q1),
                "q3": float(q3),
                "max": float(np.max(v)),
                "n": int(len(v)),
            }
            rows.append(row)
    return pd.DataFrame(rows)


def make_table_MDS_summary(eig_list: dict, stress_list: dict,
                           dist_by_ds: dict) -> pd.DataFrame:
    """
    Table 4.2-1 - Summary of MDS and stress values (A–D).
    eig_list: {"A": df, "B": df, ...} each df has columns 'dim', 'eigenvalue'
    stress_list: {"A": stress_val, ...}
    dist_by_ds: for n_samples
    """
    rows = []
    for ds_label, eig_df in eig_list.items():
        eig_df = eig_df.sort_values("dim")
        top10 = eig_df.head(10).copy()
        # pad if <10
        if top10.shape[0] < 10:
            pad_n = 10 - top10.shape[0]
            pad = pd.DataFrame({
                "dim": range(top10["dim"].max() + 1,
                             top10["dim"].max() + 1 + pad_n),
                "eigenvalue": [np.nan] * pad_n
            })
            top10 = pd.concat([top10, pad], ignore_index=True)

        neg_ratio = float(np.mean(eig_df["eigenvalue"] < 0))

        # n_samples from raw distance
        d_raw = dist_by_ds[ds_label]["raw"]
        d_arr = np.array(d_raw)
        if d_arr.ndim == 2 and d_arr.shape[0] == d_arr.shape[1]:
            n_samples = d_arr.shape[0]
        else:
            # fallback (not exact)
            n_samples = int(np.round((1 + np.sqrt(1 + 8 * len(d_arr.ravel()))) / 2))

        n_features = np.nan  # unknown from STEP1 alone

        stress_val = stress_list.get(ds_label, np.nan)

        row = {
            "dataset": ds_label,
            "n_samples": n_samples,
            "n_features": n_features,
            "used_dims": 10,
            "neg_eigen_ratio": neg_ratio,
            "isoMDS_stress": float(stress_val),
        }

        for k in range(1, 11):
            row[f"eig{k}"] = float(top10["eigenvalue"].iloc[k - 1])

        rows.append(row)

    return pd.DataFrame(rows)


def make_table_isomds_stress_full(step1_dir: str, dataset_folders: dict) -> pd.DataFrame:
    """
    Table S4.2.2 - Full isoMDS stress table (A–D).
    """
    rows = []
    for ds_label, folder in dataset_folders.items():
        stress_val = read_isomds_stress(step1_dir, ds_label, folder)
        rows.append({
            "dataset": ds_label,
            "stress": float(stress_val),
        })
    return pd.DataFrame(rows)


# =============
# 4. Main
# =============

def main():
    parser = argparse.ArgumentParser(
        description="Create STEP1 figures and tables (Distance & MDS)."
    )
    parser.add_argument(
        "--root",
        type=str,
        default="~/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127",
        help="ROOT directory (default: %(default)s)",
    )
    args = parser.parse_args()

    ROOT = os.path.expanduser(args.root)
    print(f"[INFO] ROOT = {ROOT}")

    step1_dir = os.path.join(ROOT, "data", "calc_core", "01_distance_mds")
    out_root = os.path.join(ROOT, "figures_tables", "STEP1_Distance_MDS")

    fig_main_dir = os.path.join(out_root, "figures_main")
    fig_appendix_dir = os.path.join(out_root, "figures_appendix")
    tab_main_dir = os.path.join(out_root, "tables_main")
    tab_appendix_dir = os.path.join(out_root, "tables_appendix")

    for d in [fig_main_dir, fig_appendix_dir, tab_main_dir, tab_appendix_dir]:
        os.makedirs(d, exist_ok=True)
        print(f"[INFO] Ensured directory: {d}")

    date_stamp = date.today().strftime("%Y%m%d")
    print(f"[INFO] Date stamp: {date_stamp}")

    # Dataset folders
    dataset_folders = {
        "A": "A_OH_plus_others",
        "B": "B_FP_plus_others",
        "C": "C_OH_only",
        "D": "D_FP_only",
    }

    # --- Read distances ---
    print("[INFO] Reading distance matrices (A–D)...")
    dist_by_ds = {
        ds: read_distance_pair(step1_dir, folder)
        for ds, folder in dataset_folders.items()
    }

    # =========
    # Fig. 4.2A
    # =========
    fig42A_pdf = os.path.join(
        fig_main_dir, f"Fig_4.2A_distance_distribution_{date_stamp}.pdf"
    )
    fig42A_png = os.path.join(
        fig_main_dir, f"Fig_4.2A_distance_distribution_{date_stamp}.png"
    )
    plot_distance_distribution_AB(
        dist_by_ds["A"], dist_by_ds["B"],
        outfile_pdf=fig42A_pdf,
        outfile_png=fig42A_png
    )

    # =========
    # Fig. 4.2B (cmdscale MDS A/B)
    # =========
    print("[INFO] Reading cmdscale MDS (A/B)...")
    mds_cmd_A = read_cmdscale_mds(step1_dir, "A", dataset_folders["A"])
    mds_cmd_B = read_cmdscale_mds(step1_dir, "B", dataset_folders["B"])

    fig42B_pdf = os.path.join(
        fig_main_dir, f"Fig_4.2B_cmdscale_MDSmap_{date_stamp}.pdf"
    )
    plot_cmdscale_MDS_AB(mds_cmd_A, mds_cmd_B, outfile_pdf=fig42B_pdf)

    # =========
    # Fig. 4.2C (isoMDS A/B)
    # =========
    print("[INFO] Reading isoMDS (A/B)...")
    mds_iso_A = read_isomds_mds(step1_dir, "A", dataset_folders["A"])
    mds_iso_B = read_isomds_mds(step1_dir, "B", dataset_folders["B"])

    fig42C_pdf = os.path.join(
        fig_main_dir, f"Fig_4.2C_isoMDS_MDSmap_{date_stamp}.pdf"
    )
    plot_isoMDS_AB(mds_iso_A, mds_iso_B, outfile_pdf=fig42C_pdf)

    # =========
    # Fig. 4.2D (Scree A/B)
    # =========
    print("[INFO] Reading eigenvalues (A/B)...")
    eig_A = read_cmdscale_eigvals(step1_dir, "A", dataset_folders["A"])
    eig_B = read_cmdscale_eigvals(step1_dir, "B", dataset_folders["B"])

    fig42D_pdf = os.path.join(
        fig_main_dir, f"Fig_4.2D_scree_plot_{date_stamp}.pdf"
    )
    plot_scree_AB(eig_A, eig_B, outfile_pdf=fig42D_pdf)

    # =========
    # Table 4.2-1 (MDS summary, A–D)
    # =========
    print("[INFO] Creating Table 4.2-1 (MDS summary, A–D)...")
    eig_list = {
        "A": eig_A,
        "B": eig_B,
        "C": read_cmdscale_eigvals(step1_dir, "C", dataset_folders["C"]),
        "D": read_cmdscale_eigvals(step1_dir, "D", dataset_folders["D"]),
    }

    stress_list = {
        ds: read_isomds_stress(step1_dir, ds, folder)
        for ds, folder in dataset_folders.items()
    }

    table_MDS_summary = make_table_MDS_summary(eig_list, stress_list, dist_by_ds)
    tab421_file = os.path.join(
        tab_main_dir, f"Table_4.2-1_MDS_summary_{date_stamp}.csv"
    )
    table_MDS_summary.to_csv(tab421_file, index=False)
    print(f"[SAVED] Table 4.2-1: {tab421_file}")

    # ==========================
    # Appendix figures & tables
    # ==========================

    # Scree plots for C/D
    print("[INFO] Appendix: Scree plots (C/D)...")
    eig_C = eig_list["C"]
    eig_D = eig_list["D"]

    figS421_pdf = os.path.join(
        fig_appendix_dir, f"Fig_S4.2.1_scree_plot_C_{date_stamp}.pdf"
    )
    figS422_pdf = os.path.join(
        fig_appendix_dir, f"Fig_S4.2.2_scree_plot_D_{date_stamp}.pdf"
    )
    plot_scree_single(eig_C, "C (OH only)", figS421_pdf)
    plot_scree_single(eig_D, "D (FP only)", figS422_pdf)

    # 2D MDS maps for C/D (cmdscale)
    print("[INFO] Appendix: cmdscale MDS maps (C/D)...")
    mds_cmd_C = read_cmdscale_mds(step1_dir, "C", dataset_folders["C"])
    mds_cmd_D = read_cmdscale_mds(step1_dir, "D", dataset_folders["D"])

    figS423_pdf = os.path.join(
        fig_appendix_dir, f"Fig_S4.2.3_cmdscale_MDSmap_C_{date_stamp}.pdf"
    )
    figS424_pdf = os.path.join(
        fig_appendix_dir, f"Fig_S4.2.4_cmdscale_MDSmap_D_{date_stamp}.pdf"
    )
    plot_cmdscale_single(mds_cmd_C, "C (OH only)", figS423_pdf)
    plot_cmdscale_single(mds_cmd_D, "D (FP only)", figS424_pdf)

    # Table S4.2.1: raw vs corrected distance statistics (A–D)
    print("[INFO] Appendix: Table S4.2.1 (distance stats A–D)...")
    table_dist_stats = make_table_distance_stats(dist_by_ds)
    tabS421_file = os.path.join(
        tab_appendix_dir,
        f"Table_S4.2.1_raw_vs_corrected_distance_stats_{date_stamp}.csv",
    )
    table_dist_stats.to_csv(tabS421_file, index=False)
    print(f"[SAVED] Table S4.2.1: {tabS421_file}")

    # Table S4.2.2: isoMDS stress (A–D)
    print("[INFO] Appendix: Table S4.2.2 (isoMDS stress A–D)...")
    table_isomds_stress = make_table_isomds_stress_full(step1_dir, dataset_folders)
    tabS422_file = os.path.join(
        tab_appendix_dir,
        f"Table_S4.2.2_isoMDS_stress_{date_stamp}.csv",
    )
    table_isomds_stress.to_csv(tabS422_file, index=False)
    print(f"[SAVED] Table S4.2.2: {tabS422_file}")

    print("[INFO] All STEP1 figures/tables generated.")


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'pyreadr'